# Demo 1 — Vision & Gradient Attacks
## FGSM and PGD on MNIST

**Threat Model:** White-box (attacker knows model weights & gradients)

**Goal:** Cause a trained image classifier to misclassify by adding a small,
human-imperceptible perturbation in the direction of the gradient of the loss
with respect to the *input* (not the weights).

### Key Attacks Demonstrated
| Attack | Type | Steps |
|--------|------|-------|
| FGSM   | Single-step | 1 |
| PGD    | Iterative   | k |

**Libraries used:** `torch`, `torchvision`, `matplotlib`

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## Step 1 — Define a Simple CNN
We train a small convolutional network on MNIST (28x28 grayscale digits 0-9).
This is our **victim model** — the target of the attack.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_set_f = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set_f  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_set = torch.utils.data.Subset(train_set_f, range(1000))
test_set  = torch.utils.data.Subset(test_set_f, range(200))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False)
print(f'Train: {len(train_set)} | Test: {len(test_set)}')

## Step 2 — Train the Victim Model

In [ ]:
model = SimpleCNN().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

print('Training CNN on MNIST...')
for epoch in range(3):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'  Epoch {epoch+1}/3 - Loss: {total_loss/len(train_loader):.4f}')

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            correct += (model(X).argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

clean_acc = evaluate(model, test_loader)
print(f'\nClean test accuracy: {clean_acc*100:.2f}%')

## Step 3 — FGSM Attack

**Formula:** $x_{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$

We take one step in the direction that *maximises* the loss, using the gradient
with respect to the **input pixels** (not model weights).

In [ ]:
def fgsm(model, X, y, epsilon):
    """Fast Gradient Sign Method (Goodfellow et al., 2014)."""
    X_adv = X.clone().detach().requires_grad_(True)
    loss = loss_fn(model(X_adv), y)
    loss.backward()
    perturbation = epsilon * X_adv.grad.sign()
    return (X + perturbation).clamp(0, 1).detach()

## Step 4 — PGD Attack (Stronger)

PGD repeats the FGSM step **k times**, projecting back into the epsilon-ball:

$x^{t+1} = \Pi_{\|\delta\|_\infty \leq \varepsilon}\left(x^t + \alpha \cdot \text{sign}(\nabla_x J)\right)$

Because it is iterative, PGD is far stronger than FGSM and is considered the
**gold standard** for evaluating robustness.

In [ ]:
def pgd(model, X, y, epsilon, alpha=0.01, num_steps=40):
    """Projected Gradient Descent (Madry et al., 2018)."""
    X_adv = X.clone().detach()
    for _ in range(num_steps):
        X_adv.requires_grad_(True)
        loss = loss_fn(model(X_adv), y)
        loss.backward()
        with torch.no_grad():
            X_adv = X_adv + alpha * X_adv.grad.sign()
            delta = torch.clamp(X_adv - X, -epsilon, epsilon)
            X_adv = torch.clamp(X + delta, 0, 1).detach()
    return X_adv

## Step 5 — Measure Accuracy Under Attack

In [ ]:
def attack_accuracy(model, loader, attack_fn, **kwargs):
    model.eval()
    correct = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        X_adv = attack_fn(model, X, y, **kwargs)
        correct += (model(X_adv).argmax(1) == y).sum().item()
    return correct / len(loader.dataset)

epsilons = [0.05, 0.1, 0.2, 0.3]
fgsm_accs = [attack_accuracy(model, test_loader, fgsm, epsilon=e) for e in epsilons]
pgd_accs  = [attack_accuracy(model, test_loader, pgd,  epsilon=e) for e in epsilons]

print(f'{'epsilon':>8}  {'FGSM Acc':>10}  {'PGD Acc':>10}')
print('-' * 35)
for e, fa, pa in zip(epsilons, fgsm_accs, pgd_accs):
    print(f'{e:>8.2f}  {fa*100:>9.2f}%  {pa*100:>9.2f}%')

## Step 6 — Visualise Adversarial Examples

In [ ]:
model.eval()
X_sample, y_sample = next(iter(test_loader))
X_sample, y_sample = X_sample[:8].to(DEVICE), y_sample[:8].to(DEVICE)

epsilon = 0.2
X_fgsm = fgsm(model, X_sample, y_sample, epsilon)
X_pgd_mid = pgd(model, X_sample, y_sample, epsilon, num_steps=10)
X_pgd  = pgd(model,  X_sample, y_sample, epsilon)

with torch.no_grad():
    p_clean = model(X_sample).argmax(1).cpu()
    p_fgsm  = model(X_fgsm).argmax(1).cpu()
    p_pgd_mid = model(X_pgd_mid).argmax(1).cpu()
    p_pgd   = model(X_pgd).argmax(1).cpu()
    true    = y_sample.cpu()

def save_images(imgs, preds, title, filename):
    fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
    for col in range(8):
        axes[col].imshow(imgs[col].squeeze().cpu(), cmap='gray')
        c = 'green' if preds[col]==true[col] else 'red'
        axes[col].set_title(f'P:{preds[col]}', color=c, fontsize=10)
        axes[col].axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

save_images(X_sample, p_clean, 'Before Attack (Clean)', '01_before_attack.png')
save_images(X_pgd_mid, p_pgd_mid, 'In Process Attack (PGD 10 steps)', '01_in_process_attack.png')
save_images(X_pgd, p_pgd, 'After Attack (PGD 40 steps)', '01_after_attack.png')

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle(f'Vision & Gradient Attacks (epsilon={epsilon})', fontsize=14, fontweight='bold')
rows = [(X_sample, p_clean, 'Clean'), (X_fgsm, p_fgsm, 'FGSM'), (X_pgd, p_pgd, 'PGD-40')]
for row_idx, (imgs, preds, label) in enumerate(rows):
    for col in range(8):
        axes[row_idx, col].imshow(imgs[col].squeeze().cpu(), cmap='gray')
        c = 'green' if preds[col]==true[col] else 'red'
        axes[row_idx, col].set_title(f'P:{preds[col]}', color=c, fontsize=8)
        axes[row_idx, col].axis('off')
    axes[row_idx, 0].set_ylabel(label, fontsize=10)
plt.tight_layout()
plt.savefig('01_vision_gradient_results.png', dpi=150)
plt.show()
print('Saved 01_before_attack.png, 01_in_process_attack.png, 01_after_attack.png')
print('Green = correct | Red = fooled by attack')

## Summary

| Attack | epsilon=0.2 Accuracy |
|--------|---------------------|
| Clean  | ~99%                |
| FGSM   | drops sharply       |
| PGD    | near 0%             |

**Key takeaway:** A single-step attack (FGSM) already dramatically reduces accuracy.
An iterative attack (PGD) can reduce accuracy near 0% while the perturbation
remains invisible to the human eye.

**Defenses:** Adversarial Training, Feature Squeezing, Randomised Smoothing.